# Data Cleaning with Pandas



In [31]:
import pandas as pd

In [32]:
# orders.csv
url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames.

In [33]:
orders_df = orders.copy()

In [34]:
orderlines_df = orderlines.copy()

## 1.&nbsp; Duplicates


In [35]:
# orders_df
orders_df.duplicated().sum()

np.int64(0)

In [36]:
# orderlines_df
orderlines_df.duplicated().sum()

np.int64(0)

We have no duplicate rows in either DataFrame.

# 2. .info()

In [37]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [38]:
orderlines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [39]:
num_missing = orders_df['total_paid'].isna().sum()
total_rows = orders_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"5 missing values represents {percent_missing:.5f}% of the rows in our DataFrame")

5 missing values represents 0.00220% of the rows in our DataFrame


In [40]:
orders_df['total_paid'].isna().value_counts(normalize=True)

,proportion
total_paid,
False,0.999978
True,0.000022


As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [41]:
orders_df = orders_df.dropna(axis=0)

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines_df`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [42]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [43]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [45]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

ValueError: Unable to parse string "1.137.99" at position 6

As you can see when we try to convert `unit_price` to a numerical datatype, we receive a `ValueError`

In [ ]:
# Count the number of decimal points in the unit_price
orderlines_df['unit_price'].str.count(r"\.").value_counts()

Looks like over 36000 rows in `orderlines` are affected by this problem.

In [ ]:
# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

In [ ]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# Keep only the rows that do not have multiple decimal points
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

In [ ]:
orderlines_df.shape[0]

We still have 216250 rows in orderlines to work with.

Now that all of the 2 decimal point prices have been removed

In [ ]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

It worked perfectly

#Clean the `products` DataFrame

In [ ]:
# products.csv
url = "https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

In [ ]:
products_df = products.copy()

In [ ]:
# your code here
products.shape

In [ ]:
products.head()

### Look for Duplicates

In [ ]:
# your code here
products.duplicated().sum()

In [ ]:
products_df = products_df.drop_duplicates()

In [ ]:
products_df.duplicated().sum()

### Look for Missing values


In [ ]:

products.isna().sum()

In [ ]:
(products.isna().mean() * 100).round(2)

In [ ]:
products_df = products_df.dropna()

In [ ]:
products_df.isna().sum()

### Check / Change Data types

In [ ]:
# your code here
products_df['price'].str.count(r"\.").value_counts()

In [ ]:
mult_decimal_rows = (products_df["price"].str.count(r"\.") > 1).sum()

percent_corrupted = (100 * mult_decimal_rows / products_df.shape[0])

print(f"{percent_corrupted:.2f}% of the rows in the products dataset have multiple decimal points in the price column.")

In [ ]:
products_df = products_df[products_df["price"].str.count(r"\.") <= 1]

In [ ]:
products_df.shape

In [ ]:
products_df["price"] = pd.to_numeric(products_df["price"])
products_df.dtypes

In [ ]:
products_df = products_df.drop(columns=["promo_price"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Project Eniac/"
products_df.to_csv(PATH + "products_clean.csv", index=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Project Eniac/"
orders_df.to_csv(PATH + "orders_clean.csv", index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Project Eniac/"
orderlines_df.to_csv(PATH + "orders_line_clean.csv", index=False)